# Dongle Push Demo

In [ ]:
import random

from pyzx.pauliweb import compute_pauli_webs

random.seed(50)

import pyzx as zx

g = zx.generate.cnots(4, 5)
zx.id_simp(g)

In [ ]:
from dongle import ShieldedGraph

bg = ShieldedGraph.from_graph(g)
x_dongle, z_dongle, y_dongle = bg.add_dongles((9, 12))

bgc = bg.clone(ShieldedGraph())
bgc.full_instance()
zx.draw(bgc, labels=True)

In [ ]:
from dongle import compute_webs_for_dongle, fire_web_onto_dongle

debug = dict()
webs = None
try:
    webs = compute_webs_for_dongle(graph=bg, dongle_id=y_dongle.get_id(), debug=debug)
except Exception as e:
    debug['g'].pack_circuit_rows()
    for i, web in enumerate(debug['g_webs']):
        zx.draw(debug['g'], pauli_web=web, labels=True)
    raise e
    
debug['g'].pack_circuit_rows()
bgc.pack_circuit_rows()

for i, web in enumerate(webs):
    bgg = bg.clone(ShieldedGraph())
    
    print("Internal firing web:")
    zx.draw(debug['g'], pauli_web=debug['relevant_g_webs'][i], labels=True)
    print("Firing web mapped onto original graph:")
    zx.draw(bgc, pauli_web=web, labels=True)
    fire_web_onto_dongle(g=bgg, dongle_id=y_dongle.get_id(), web=web)
    bgc1 = bgg.clone(ShieldedGraph())
    bgc1.full_instance()
    print("Graph with fired web for dongle:")
    zx.draw(bgc1, labels=True)
    bgg.merge_targets()
    bgc2 = bgg.clone(ShieldedGraph())
    bgc2.full_instance()
    print("Merged dongle:")
    zx.draw(bgc2, labels=True)
    print("Firing finished!")

In [ ]:
from dongle import expand_all_dongles

bgg = bg.clone(ShieldedGraph())
bgg.reassign_dongle_positions()
bgg.pack_circuit_rows()
expand_all_dongles(bgg)
bgg.full_instance()
zx.draw(bgg, labels=True)